# 다중 표현 검색(Multi-vector / multi-representation retrieval)

검색에 최적화된 여러 표현(작은 청크, 요약, 가설 질문)을 임베딩하되, 최종
응답에는 원문을 반환합니다. classic의 `MultiVectorRetriever` 없이 Chroma의
표현 인덱스와 명시적인 부모 저장소를 조합합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" \
#   "langchain-text-splitters==1.1.2" python-dotenv pymupdf


In [ ]:
import getpass
import os
from pathlib import Path
from uuid import uuid4

import pymupdf
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
model = init_chat_model(f"openai:{model_name}", temperature=0)
CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


In [ ]:
def load_pdf_pages(path: Path) -> list[Document]:
    if not path.exists():
        raise FileNotFoundError(f"실습 PDF가 없습니다: {path.resolve()}")
    with pymupdf.open(path) as pdf:
        return [
            Document(
                page_content=page.get_text("text"),
                metadata={"source": str(path), "page": page.number},
            )
            for page in pdf
            if page.get_text("text").strip()
        ]


pdf_path = Path("data/SPRI_AI_Brief_2023년12월호_F.pdf")
pages = load_pdf_pages(pdf_path)
print(f"로드한 PDF 페이지 수: {len(pages)}")
print(pages[min(5, len(pages) - 1)].page_content[:500])


## 공통 다중 표현 검색기


In [ ]:
def build_multi_representation_index(
    representations: list[Document],
    parents: list[Document],
    parent_ids: list[str],
    *,
    k: int = 4,
    name: str,
) -> tuple[Chroma, RunnableLambda]:
    if len(parents) != len(parent_ids):
        raise ValueError("parents와 parent_ids의 길이가 같아야 합니다.")
    parent_store = dict(zip(parent_ids, parents))

    vectorstore = Chroma.from_documents(
        representations,
        embeddings,
        collection_name=f"{name}-{uuid4().hex}",
        ids=[uuid4().hex for _ in representations],
        collection_configuration=CHROMA_CONFIGURATION,
    )

    def retrieve(query: str) -> list[Document]:
        hits = vectorstore.similarity_search(query, k=k)
        ordered_ids: list[str] = []
        for hit in hits:
            parent_id = hit.metadata["parent_id"]
            if parent_id not in ordered_ids:
                ordered_ids.append(parent_id)
        return [parent_store[parent_id] for parent_id in ordered_ids]

    return vectorstore, RunnableLambda(retrieve).with_config(
        {"run_name": f"{name}_retriever"}
    )


## 1. 작은 청크를 검색하고 원본 페이지 반환

각 자식 청크의 `parent_id`가 실제 원본 페이지를 가리키게 합니다. 부모·자식을
원문에서 따로 쪼개 같은 ID를 붙이는 오래된 예제의 매핑 오류를 피합니다.


In [ ]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    add_start_index=True,
)
page_ids = [uuid4().hex for _ in pages]
child_representations: list[Document] = []
for page, parent_id in zip(pages, page_ids):
    for child in child_splitter.split_documents([page]):
        child_representations.append(
            Document(
                page_content=child.page_content,
                metadata={**child.metadata, "parent_id": parent_id},
            )
        )

child_vectorstore, child_to_page = build_multi_representation_index(
    child_representations,
    pages,
    page_ids,
    k=6,
    name="child-to-page",
)
query = "삼성전자가 만든 생성형 AI의 이름은?"
print("검색 표현:", child_vectorstore.similarity_search(query, k=1)[0].page_content)
print("\n반환 원문:\n", child_to_page.invoke(query)[0].page_content)


## 2. 요약을 검색 표현으로 사용

아래 기본값은 API 비용을 제한하기 위해 앞의 40개 청크만 처리합니다.
전체 문서를 인덱싱하려면 `RAG_DEMO_LIMIT`을 늘리되 배치 비용과 rate limit을
먼저 확인하세요.


In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    add_start_index=True,
)
split_docs = parent_splitter.split_documents(pages)
demo_limit = int(os.getenv("RAG_DEMO_LIMIT", "40"))
demo_docs = split_docs[:demo_limit]
demo_parent_ids = [uuid4().hex for _ in demo_docs]

summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "문서의 검색용 한국어 요약을 정확히 세 문장으로 작성하세요."),
        ("user", "{document}"),
    ]
)
summary_chain = summary_prompt | model | StrOutputParser()
summaries = summary_chain.batch(
    [{"document": doc.page_content} for doc in demo_docs],
    config={"max_concurrency": 5},
)
summary_docs = [
    Document(page_content=summary, metadata={"parent_id": parent_id})
    for summary, parent_id in zip(summaries, demo_parent_ids)
]

summary_vectorstore, summary_retriever = build_multi_representation_index(
    summary_docs,
    demo_docs,
    demo_parent_ids,
    k=4,
    name="summary-to-original",
)
print("검색된 요약:\n", summary_vectorstore.similarity_search(query, k=1)[0].page_content)
print("\n반환 원문:\n", summary_retriever.invoke(query)[0].page_content)


## 3. 가설 질문을 검색 표현으로 사용

예전 `functions=` 바인딩과 `JsonKeyOutputFunctionsParser` 대신 Pydantic 스키마를
`with_structured_output()`에 전달합니다.


In [ ]:
class HypotheticalQuestions(BaseModel):
    questions: list[str] = Field(
        min_length=3,
        max_length=3,
        description="이 문서로 답할 수 있는 서로 다른 한국어 질문 3개",
    )


question_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "AI 산업에 관심 있는 독자가 아래 문서에 물을 법한 질문을 정확히 "
            "3개 생성하세요. 문서 밖의 사실을 만들지 마세요.",
        ),
        ("user", "{document}"),
    ]
)
hypothetical_chain = question_prompt | model.with_structured_output(
    HypotheticalQuestions
)
question_sets = hypothetical_chain.batch(
    [{"document": doc.page_content} for doc in demo_docs],
    config={"max_concurrency": 5},
)

question_docs: list[Document] = []
for output, parent_id in zip(question_sets, demo_parent_ids):
    question_docs.extend(
        Document(page_content=question, metadata={"parent_id": parent_id})
        for question in output.questions
    )

question_vectorstore, question_retriever = build_multi_representation_index(
    question_docs,
    demo_docs,
    demo_parent_ids,
    k=6,
    name="questions-to-original",
)
print("가장 가까운 가설 질문:")
for doc in question_vectorstore.similarity_search(query, k=3):
    print("-", doc.page_content)
print("\n반환 원문:\n", question_retriever.invoke(query)[0].page_content)
